# Phase 9-C Step 3: 4システム比較評価

**作成日**: 2026-03-01  
**プロジェクト**: experiments-local-llm  
**目的**: FT単体・RAG単体・FT+RAG・FT-bare の効果を分離測定する

---

## 比較する 4 システム

| System | モデル | プロンプト | RAG | 目的 |
|--------|--------|-----------|-----|------|
| **A: RAG (C2)** | Qwen3-32B | C1 改善版 | あり | ベースライン（既存結果） |
| **B: FT+prompt** | QLoRA FT | C1 改善版 | なし | FT + 改善プロンプトの効果 |
| **C: FT+RAG** | QLoRA FT | C1 改善版 | あり | FT + RAG 相補効果 |
| **D: FT-bare** | QLoRA FT | **C0 オリジナル** | なし | FT 単体（プロンプト改善なし） |

## C3 目標値

| 指標 | C2 実績 | C3 目標 |
|------|---------|--------|
| composite_score | 70.4 | **75+** |
| reasoning_score | 3.07 | **3.5+** |
| evidence_score | 3.85 | **4.0+** |
| composite_success_rate | 83.1% | **88%+** |

## 0. リポジトリ同期

In [1]:
import os
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')

REPO_PATH = '/content/drive/MyDrive/experiments-local-llm'
BRANCH = 'feature/phase9c-step3-qlora'

print(f"Repository: {REPO_PATH}")
print(f"Target branch: {BRANCH}")
print()

from getpass import getpass
github_token = getpass("GitHub Personal Access Token: ")
!git -C {REPO_PATH} remote set-url origin https://{github_token}@github.com/mopinfish/experiments-local-llm.git
print("GitHub HTTPS authentication configured\n")

!git -C {REPO_PATH} config user.name "colab-runner"
!git -C {REPO_PATH} config user.email "colab@example.com"
!git -C {REPO_PATH} stash --include-untracked -m "auto-stash before sync"
!git -C {REPO_PATH} fetch origin
!git -C {REPO_PATH} checkout {BRANCH}
!git -C {REPO_PATH} pull origin {BRANCH}
!git -C {REPO_PATH} stash pop 2>/dev/null || echo "No stash to pop"

print()
print("=" * 60)
!git -C {REPO_PATH} log --oneline -5
!git -C {REPO_PATH} branch --show-current
print("=" * 60)
print("\n\u2705 リポジトリ同期完了")

Mounted at /content/drive
Repository: /content/drive/MyDrive/experiments-local-llm
Target branch: feature/phase9c-step3-qlora

GitHub Personal Access Token: ··········
GitHub HTTPS authentication configured

Saved working directory and index state On phase9c-step3-qlora: auto-stash before sync
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 3.61 KiB | 7.00 KiB/s, done.
From https://github.com/mopinfish/experiments-local-llm
   71d0f90..232d408  feature/phase9c-step3-qlora -> origin/feature/phase9c-step3-qlora
Already on 'feature/phase9c-step3-qlora'
Your branch is behind 'origin/feature/phase9c-step3-qlora' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/mopinfish/experiments-local-llm
 * branch            feature/phase9c-step3-qlora -> FETCH_HEAD
Updating 71d0f90..232d408
Fast-forward
 note

## 1. 環境セットアップ

In [2]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    PROJECT_PATH = '/content/drive/MyDrive/experiments-local-llm'
    sys.path.insert(0, f'{PROJECT_PATH}/src')

    !pip install -q chromadb sentence-transformers networkx
    !pip install -q transformers accelerate bitsandbytes
    !pip install -q peft
    !pip install -q langchain langchain-core langchain-community langchain-chroma langgraph
    !pip install -q matplotlib seaborn pandas numpy tqdm
else:
    PROJECT_PATH = '..'
    sys.path.insert(0, f'{PROJECT_PATH}/src')

os.makedirs(f'{PROJECT_PATH}/results', exist_ok=True)

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")

    if 'A100' not in gpu_name:
        print(f"\n\u26a0\ufe0f WARNING: A100 GPU required!")
        print(f"  Current GPU: {gpu_name}")
        raise RuntimeError(f"A100 GPU required, got {gpu_name}")
    else:
        print("\u2705 A100 GPU confirmed")

print(f"\nProject path: {PROJECT_PATH}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 128.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curre

## 2. FTモデルロード (Qwen3-32B + LoRA)

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import gc
import re
import json
import warnings
warnings.filterwarnings('ignore')

# VRAMクリーンアップ
for var_name in ['model', 'ft_model', 'tokenizer']:
    if var_name in dir():
        try:
            del globals()[var_name]
        except KeyError:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM before load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# ベースモデルロード
model_name = "Qwen/Qwen3-32B"
print(f"Loading base model: {model_name}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print("Tokenizer loaded")

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
print("Base model loaded")

# LoRAアダプターロード
adapter_path = "/content/drive/MyDrive/models/qwen3-32b-poi-qlora"
print(f"\nLoading LoRA adapter from: {adapter_path}")
ft_model = PeftModel.from_pretrained(base_model, adapter_path)
ft_model.eval()
print("LoRA adapter loaded")

# Qwen3 thinking mode 無効化
original_apply = tokenizer.apply_chat_template
def patched_apply(*args, **kwargs):
    kwargs['enable_thinking'] = False
    return original_apply(*args, **kwargs)
tokenizer.apply_chat_template = patched_apply
print("\u2705 Qwen3 thinking mode disabled")

if torch.cuda.is_available():
    print(f"\nVRAM after FT model load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# 学習メタデータ確認
meta_path = f"{adapter_path}/training_metadata.json"
if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    print(f"\nTraining metadata:")
    print(f"  Train samples: {meta.get('train_samples')}")
    print(f"  Epochs: {meta.get('num_train_epochs')}")
    print(f"  Best eval loss: {meta.get('eval_loss_best')}")

VRAM before load: 0.00 GB
Loading base model: Qwen/Qwen3-32B...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Base model loaded

Loading LoRA adapter from: /content/drive/MyDrive/models/qwen3-32b-poi-qlora
LoRA adapter loaded
✅ Qwen3 thinking mode disabled

VRAM after FT model load: 18.44 GB

Training metadata:
  Train samples: 47
  Epochs: 5
  Best eval loss: 0.3545990288257599


## 3. POIデータ・ベクトルストア・テストケース読み込み

In [4]:
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model_name = "intfloat/multilingual-e5-base"
print(f"Loading embedding model: {embedding_model_name}...")

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("Embedding model loaded")

Loading embedding model: intfloat/multilingual-e5-base...


/tmp/ipython-input-8308/2337872953.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding model loaded


In [5]:
from geo_utils import STATIONS, enrich_all_areas

# エリア設定
areas_config = {
    "shibuya": {"name": "渋谷駅周辺", "station": STATIONS["渋谷駅"]},
    "shinjuku": {"name": "新宿駅周辺", "station": STATIONS["新宿駅"]},
    "ikebukuro": {"name": "池袋駅周辺", "station": STATIONS["池袋駅"]},
    "tokyo": {"name": "東京駅周辺", "station": STATIONS["東京駅"]},
}

# POIデータ読み込み
poi_all_file = f"{PROJECT_PATH}/data/poi_all_areas.json"
print(f"Loading POI data from {poi_all_file}...")

with open(poi_all_file, "r", encoding="utf-8") as f:
    raw_pois = json.load(f)

flat_pois = []
for poi in raw_pois:
    if "metadata" in poi:
        flat_pois.append(poi["metadata"].copy())
    else:
        flat_pois.append(poi)

print(f"Total POIs: {len(flat_pois)}")

# 空間情報付与
all_pois_enriched = enrich_all_areas(flat_pois, areas_config)
print(f"Enriched {len(all_pois_enriched)} POIs")

Loading POI data from /content/drive/MyDrive/experiments-local-llm/data/poi_all_areas.json...
Total POIs: 3567
Enriched 3567 POIs


In [6]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

def create_documents(pois):
    docs = []
    for poi in pois:
        content = f"{poi.get('name', '')} {poi.get('category', '')} {poi.get('description', '')}"
        docs.append(Document(page_content=content, metadata=poi))
    return docs

# 全統合ベクトルストア
all_docs = create_documents(flat_pois)
vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    collection_name="pois_all"
)
print(f"Vectorstore: {len(all_docs)} documents")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Vectorstore: 3567 documents


In [7]:
from test_cases_multi_area import ALL_MULTI_AREA_TEST_CASES, get_quick_test_cases

print(f"Total test cases: {len(ALL_MULTI_AREA_TEST_CASES)}")

# 学習データのtest_idセットを読み込み（過学習分析用）
training_data_file = f"{PROJECT_PATH}/data/phase9c_training_data.json"
with open(training_data_file, encoding="utf-8") as f:
    training_data = json.load(f)

train_test_ids = set(s["metadata"]["test_id"] for s in training_data["train"])
valid_test_ids = set(s["metadata"]["test_id"] for s in training_data["validation"])
ft_data_ids = train_test_ids | valid_test_ids

print(f"FT data test IDs: {len(ft_data_ids)} (train={len(train_test_ids)}, valid={len(valid_test_ids)})")
print(f"Non-FT test IDs: {len(ALL_MULTI_AREA_TEST_CASES) - len(ft_data_ids)}")

Total test cases: 130
FT data test IDs: 59 (train=47, valid=12)
Non-FT test IDs: 71


## 4. System A: RAG (C2) — 既存結果読み込み

In [8]:
# C2結果読み込み
c2_results_file = f"{PROJECT_PATH}/results/phase9c_step2_20260227_071313.json"
print(f"Loading System A (C2 RAG) results from: {c2_results_file}")

with open(c2_results_file, encoding="utf-8") as f:
    c2_data = json.load(f)

system_a_raw = c2_data["systems"]["hybrid_rag"]["results"]
system_a_summary = c2_data["systems"]["hybrid_rag"]["summary"]

print(f"System A: {len(system_a_raw)} results loaded")
print(f"  Composite: {system_a_summary['overall']['avg_composite_score']}")
print(f"  Reasoning: {system_a_summary['overall']['avg_reasoning_score']}")
print(f"  Evidence: {system_a_summary['overall']['avg_evidence_score']}")
print(f"  Success%: {system_a_summary['overall']['composite_success_rate']*100:.1f}%")

Loading System A (C2 RAG) results from: /content/drive/MyDrive/experiments-local-llm/results/phase9c_step2_20260227_071313.json
System A: 130 results loaded
  Composite: 70.4
  Reasoning: 3.07
  Evidence: 3.85
  Success%: 83.1%


## 5. System B: FT-only（RAGなし）

In [9]:
from evaluators_multi_area import MultiAreaEvaluator

evaluator = MultiAreaEvaluator(areas_config=areas_config, all_pois=all_pois_enriched)

# System B: FT-only (システムプロンプト + 質問のみ、RAGなし)
SYSTEM_PROMPT = """あなたは東京都内の主要駅周辺エリア（渋谷駅周辺、新宿駅周辺、池袋駅周辺、東京駅周辺）の地理情報に詳しいアシスタントです。
提供されたデータに基づいて、以下の構造で回答してください。

# 回答の構造
1. **結論**: 質問への直接的な回答を最初に述べる
2. **根拠**: データから得られた具体的な証拠を引用する
3. **補足**: 注意点や不確実な点があれば述べる

# 回答ルール
- 推論過程を明示する: 「したがって」「比較すると」「分析すると」「なぜなら」等の論理接続詞を使い、結論に至る過程を示す
- 根拠を具体的に引用する: POI名、座標(緯度, 経度)、距離(m)、件数を提供データから引用し、「データから」「検索結果に基づき」等で出典を明記する
- 数値は単位付きで示す: 距離はm、件数は件、座標は(35.xxx, 139.xxx)の形式で記載する
- 比較表現を使う: 「より多い」「最も近い」「〜倍」等の比較表現で差異を明確にする
- 不確実性を正直に示す: データで確認できない点は「ただし」「データの限界として」「可能性があります」「データからは確認できません」等で明記する
- 情報がない場合は「提供データからは確認できません」と正直に回答する"""


def system_b_fn(question: str) -> dict:
    """System B: FT model + system prompt only (no RAG)"""
    from geo_utils import detect_target_area

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
    answer = re.sub(r'</think>', '', answer).strip()

    del inputs, outputs
    torch.cuda.empty_cache()

    detected = detect_target_area(question, areas_config)
    return {"answer": answer, "detected_area": detected}


print("System B (FT-only) function defined")
print(f"System prompt length: {len(SYSTEM_PROMPT)} chars")

System B (FT-only) function defined
System prompt length: 543 chars


## 5.5. System D: FT-bare（C0オリジナルプロンプト、RAGなし）

In [10]:
# System D: FT-bare (C0 オリジナルプロンプト、RAGなし)
# C1改善プロンプトの寄与とFTの寄与を分離測定するための追加システム

C0_SYSTEM_PROMPT = """あなたは東京都内の主要駅周辺エリア（渋谷駅周辺、新宿駅周辺、池袋駅周辺、東京駅周辺）の地理情報に詳しいアシスタントです。
提供された情報に基づいて、正確かつ簡潔に回答してください。
座標情報がある場合は必ず含めてください。
数値データがある場合は具体的な数字を使って回答してください。
情報がない場合は「情報がありません」と正直に回答してください。"""


def system_d_fn(question: str) -> dict:
    """System D: FT model + C0 original prompt only (no RAG)"""
    from geo_utils import detect_target_area

    messages = [
        {"role": "system", "content": C0_SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
    answer = re.sub(r'</think>', '', answer).strip()

    del inputs, outputs
    torch.cuda.empty_cache()

    detected = detect_target_area(question, areas_config)
    return {"answer": answer, "detected_area": detected}


print("System D (FT-bare) function defined")
print(f"C0 system prompt length: {len(C0_SYSTEM_PROMPT)} chars")
print(f"\nC0 prompt:\n{C0_SYSTEM_PROMPT}")

System D (FT-bare) function defined
C0 system prompt length: 174 chars

C0 prompt:
あなたは東京都内の主要駅周辺エリア（渋谷駅周辺、新宿駅周辺、池袋駅周辺、東京駅周辺）の地理情報に詳しいアシスタントです。
提供された情報に基づいて、正確かつ簡潔に回答してください。
座標情報がある場合は必ず含めてください。
数値データがある場合は具体的な数字を使って回答してください。
情報がない場合は「情報がありません」と正直に回答してください。


In [11]:
# Quick test: System D
test_q = ALL_MULTI_AREA_TEST_CASES[0].prompt
print(f"Quick test Q: {test_q}")
result_d = system_d_fn(test_q)
print(f"Quick test A: {result_d['answer'][:200]}...")
print(f"Detected area: {result_d['detected_area']}")

Quick test Q: 渋谷駅の場所を教えてください
Quick test A: 渋谷駅は、東京都渋谷区渋谷2丁目にある主要な鉄道駅です。座標は **35.6895° N, 139.6917° E** です。...
Detected area: shibuya


In [12]:
# System D: Full evaluation (130 cases)
print("="*60)
print("System D: FT-bare Evaluation (130 cases)")
print("="*60)

checkpoint_d = f"{PROJECT_PATH}/results/checkpoint_c3_ft_bare.json"
results_d = evaluator.evaluate_all(
    system_name="C3_ft_bare",
    system_fn=system_d_fn,
    test_cases=ALL_MULTI_AREA_TEST_CASES,
    checkpoint_file=checkpoint_d
)

# スコア再計算
results_d = evaluator.recalculate_scores(results_d, ALL_MULTI_AREA_TEST_CASES)

summary_d = evaluator.generate_summary(results_d)
od = summary_d["overall"]
print(f"\nSystem D Results:")
print(f"  Composite:  {od['avg_composite_score']:.1f}")
print(f"  Reasoning:  {od['avg_reasoning_score']:.2f}")
print(f"  Evidence:   {od['avg_evidence_score']:.2f}")
print(f"  Success%:   {od['composite_success_rate']*100:.1f}%")
print(f"  Avg Time:   {od['avg_time_sec']:.1f}s")

gc.collect()
torch.cuda.empty_cache()
print("\n\u2705 System D evaluation complete")

System D: FT-bare Evaluation (130 cases)
  [1/130] MA-SBY-L1-01: 渋谷駅の場所を教えてください...
  [2/130] MA-SBY-L1-02: 渋谷駅周辺のコンビニを教えてください...
  [3/130] MA-SBY-L1-03: 渋谷駅周辺のスターバックスはありますか？...
  [4/130] MA-SBY-L1-04: 渋谷駅近くのカフェはありますか？...
  [5/130] MA-SBY-L2-01: 渋谷駅に最も近いコンビニはどれですか？距離も推定してください...
  [6/130] MA-SBY-L2-02: 渋谷駅周辺のカフェとバー、どちらが多いですか？...
  [7/130] MA-SBY-L2-03: 渋谷駅の東側と西側、どちらにカフェが多いですか？...
  [8/130] MA-SBY-L2-04: 渋谷ヒカリエから最も近いカフェはどこですか？...
  [9/130] MA-SBY-L3-01: 渋谷駅周辺で24時間営業のコンビニはありますか？...
  [10/130] MA-SBY-L3-02: 渋谷駅から500m以内で、電話番号がわかるカフェを教えてください...
    [checkpoint] 10 done, VRAM: 19.48 GB
  [11/130] MA-SBY-L3-03: 渋谷駅周辺のドトールを全て教えてください...
  [12/130] MA-SBY-L3-04: 渋谷109の近くでランチができるレストランは？...
  [13/130] MA-SBY-L4-01: 渋谷駅から近い順にカフェを3つ教えてください...
  [14/130] MA-SBY-L4-02: 渋谷駅周辺でカフェとコンビニが両方近い場所は？...
  [15/130] MA-SBY-L4-03: ハチ公像の周辺300mにある飲食店を教えてください...
  [16/130] MA-SBY-L4-04: 渋谷駅500m圏と1km圏でカフェの件数はどう変わりますか？...
  [17/130] MA-SBY-L5-01: 渋谷駅から最も近いカフェと、そこから300m以内の他カフェ数は？...
  [18/130] MA-SBY-L5-02: 渋谷駅周辺でコンビニの

In [13]:
# Quick test: System B
test_q = ALL_MULTI_AREA_TEST_CASES[0].prompt
print(f"Quick test Q: {test_q}")
result_b = system_b_fn(test_q)
print(f"Quick test A: {result_b['answer'][:200]}...")
print(f"Detected area: {result_b['detected_area']}")

Quick test Q: 渋谷駅の場所を教えてください
Quick test A: 【結論】  
渋谷駅は、東京都渋谷区宇田川町に位置しています。

【根拠】  
検索結果に「渋谷駅」という名称が4件表示されており、特に「渋谷駅（東急）」や「渋谷駅（小田急）」といった交通機関別の情報が含まれています。また、地理情報から「渋谷駅の座標は(35.6895, 139.6917)」と推定されます。これは、渋谷区の中心部に位置し、東京メトロ、JR東日本、小田急電鉄、東急電鉄など複数の交通機...
Detected area: shibuya


In [ ]:
# System B: Full evaluation (130 cases)
print("="*60)
print("System B: FT-only Evaluation (130 cases)")
print("="*60)

checkpoint_b = f"{PROJECT_PATH}/results/checkpoint_c3_ft_only.json"
results_b = evaluator.evaluate_all(
    system_name="C3_ft_only",
    system_fn=system_b_fn,
    test_cases=ALL_MULTI_AREA_TEST_CASES,
    checkpoint_file=checkpoint_b
)

# スコア再計算
results_b = evaluator.recalculate_scores(results_b, ALL_MULTI_AREA_TEST_CASES)

summary_b = evaluator.generate_summary(results_b)
ob = summary_b["overall"]
print(f"\nSystem B Results:")
print(f"  Composite:  {ob['avg_composite_score']:.1f}")
print(f"  Reasoning:  {ob['avg_reasoning_score']:.2f}")
print(f"  Evidence:   {ob['avg_evidence_score']:.2f}")
print(f"  Success%:   {ob['composite_success_rate']*100:.1f}%")
print(f"  Avg Time:   {ob['avg_time_sec']:.1f}s")

gc.collect()
torch.cuda.empty_cache()
print("\n\u2705 System B evaluation complete")

System B: FT-only Evaluation (130 cases)
  [1/130] MA-SBY-L1-01: 渋谷駅の場所を教えてください...
  [2/130] MA-SBY-L1-02: 渋谷駅周辺のコンビニを教えてください...
  [3/130] MA-SBY-L1-03: 渋谷駅周辺のスターバックスはありますか？...
  [4/130] MA-SBY-L1-04: 渋谷駅近くのカフェはありますか？...
  [5/130] MA-SBY-L2-01: 渋谷駅に最も近いコンビニはどれですか？距離も推定してください...
  [6/130] MA-SBY-L2-02: 渋谷駅周辺のカフェとバー、どちらが多いですか？...
  [7/130] MA-SBY-L2-03: 渋谷駅の東側と西側、どちらにカフェが多いですか？...
  [8/130] MA-SBY-L2-04: 渋谷ヒカリエから最も近いカフェはどこですか？...
  [9/130] MA-SBY-L3-01: 渋谷駅周辺で24時間営業のコンビニはありますか？...
  [10/130] MA-SBY-L3-02: 渋谷駅から500m以内で、電話番号がわかるカフェを教えてください...
    [checkpoint] 10 done, VRAM: 19.48 GB
  [11/130] MA-SBY-L3-03: 渋谷駅周辺のドトールを全て教えてください...
  [12/130] MA-SBY-L3-04: 渋谷109の近くでランチができるレストランは？...
  [13/130] MA-SBY-L4-01: 渋谷駅から近い順にカフェを3つ教えてください...
  [14/130] MA-SBY-L4-02: 渋谷駅周辺でカフェとコンビニが両方近い場所は？...
  [15/130] MA-SBY-L4-03: ハチ公像の周辺300mにある飲食店を教えてください...
  [16/130] MA-SBY-L4-04: 渋谷駅500m圏と1km圏でカフェの件数はどう変わりますか？...
  [17/130] MA-SBY-L5-01: 渋谷駅から最も近いカフェと、そこから300m以内の他カフェ数は？...
  [18/130] MA-SBY-L5-02: 渋谷駅周辺でコンビニの

## 6. System C: FT+RAG

In [ ]:
from structured_rag_system import StructuredRAGSystem
from geo_utils import detect_target_area

# FTモデルをRAGシステムに注入
print("Initializing System C (FT+RAG)...")

rag_system = StructuredRAGSystem(
    model=ft_model,
    tokenizer=tokenizer,
    vectorstore=vectorstore,
    all_pois=all_pois_enriched,
    areas_config=areas_config,
    debug=False
)
print("StructuredRAGSystem initialized with FT model")


def system_c_fn(question: str) -> dict:
    """System C: FT model + RAG"""
    result = rag_system.query(question)
    answer = result.get("answer", "")
    answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
    answer = re.sub(r'</think>', '', answer).strip()

    detected = result.get("target_area") or detect_target_area(question, areas_config)
    return {"answer": answer, "detected_area": detected}


print("System C (FT+RAG) function defined")

Initializing System C (FT+RAG)...
StructuredRAGSystem initialized with FT model
System C (FT+RAG) function defined


In [ ]:
# Quick test: System C
test_q = ALL_MULTI_AREA_TEST_CASES[0].prompt
print(f"Quick test Q: {test_q}")
result_c = system_c_fn(test_q)
print(f"Quick test A: {result_c['answer'][:200]}...")
print(f"Detected area: {result_c['detected_area']}")

Quick test Q: 渋谷駅の場所を教えてください
Quick test A: 【結論】  
渋谷駅は東京都渋谷区にある主要な交通拠点です。

【根拠】  
提供された検索結果には「渋谷 交通/鉄道駅」という情報が5回繰り返し記載されており、これは渋谷駅が交通/鉄道駅カテゴリに属するPOIであることを示しています。ただし、具体的な座標や距離(m)、件数等の数値情報は含まれていません。一般的な地理知識から、渋谷駅の座標は(35.6895, 139.6917)と推定されます。

...
Detected area: shibuya


In [ ]:
# System C: Full evaluation (130 cases)
print("="*60)
print("System C: FT+RAG Evaluation (130 cases)")
print("="*60)

checkpoint_c = f"{PROJECT_PATH}/results/checkpoint_c3_ft_rag.json"
results_c = evaluator.evaluate_all(
    system_name="C3_ft_rag",
    system_fn=system_c_fn,
    test_cases=ALL_MULTI_AREA_TEST_CASES,
    checkpoint_file=checkpoint_c
)

# スコア再計算
results_c = evaluator.recalculate_scores(results_c, ALL_MULTI_AREA_TEST_CASES)

summary_c = evaluator.generate_summary(results_c)
oc = summary_c["overall"]
print(f"\nSystem C Results:")
print(f"  Composite:  {oc['avg_composite_score']:.1f}")
print(f"  Reasoning:  {oc['avg_reasoning_score']:.2f}")
print(f"  Evidence:   {oc['avg_evidence_score']:.2f}")
print(f"  Success%:   {oc['composite_success_rate']*100:.1f}%")
print(f"  Avg Time:   {oc['avg_time_sec']:.1f}s")

gc.collect()
torch.cuda.empty_cache()
print("\n\u2705 System C evaluation complete")

System C: FT+RAG Evaluation (130 cases)
  [1/130] MA-SBY-L1-01: 渋谷駅の場所を教えてください...
  [2/130] MA-SBY-L1-02: 渋谷駅周辺のコンビニを教えてください...
  [3/130] MA-SBY-L1-03: 渋谷駅周辺のスターバックスはありますか？...
  [4/130] MA-SBY-L1-04: 渋谷駅近くのカフェはありますか？...
  [5/130] MA-SBY-L2-01: 渋谷駅に最も近いコンビニはどれですか？距離も推定してください...
  [6/130] MA-SBY-L2-02: 渋谷駅周辺のカフェとバー、どちらが多いですか？...
  [7/130] MA-SBY-L2-03: 渋谷駅の東側と西側、どちらにカフェが多いですか？...
  [8/130] MA-SBY-L2-04: 渋谷ヒカリエから最も近いカフェはどこですか？...
  [9/130] MA-SBY-L3-01: 渋谷駅周辺で24時間営業のコンビニはありますか？...
  [10/130] MA-SBY-L3-02: 渋谷駅から500m以内で、電話番号がわかるカフェを教えてください...
    [checkpoint] 10 done, VRAM: 19.48 GB
  [11/130] MA-SBY-L3-03: 渋谷駅周辺のドトールを全て教えてください...
  [12/130] MA-SBY-L3-04: 渋谷109の近くでランチができるレストランは？...
  [13/130] MA-SBY-L4-01: 渋谷駅から近い順にカフェを3つ教えてください...
  [14/130] MA-SBY-L4-02: 渋谷駅周辺でカフェとコンビニが両方近い場所は？...
  [15/130] MA-SBY-L4-03: ハチ公像の周辺300mにある飲食店を教えてください...
  [16/130] MA-SBY-L4-04: 渋谷駅500m圏と1km圏でカフェの件数はどう変わりますか？...
  [17/130] MA-SBY-L5-01: 渋谷駅から最も近いカフェと、そこから300m以内の他カフェ数は？...
  [18/130] MA-SBY-L5-02: 渋谷駅周辺でコンビニの競

## 7. 4システム比較分析

In [ ]:
import numpy as np

# System A のサマリーを直接使用
oa = system_a_summary["overall"]

# === 全体比較表 ===
print("="*120)
print("4 System Comparison: Overall")
print("="*120)

metrics = [
    ("Composite Score", oa['avg_composite_score'], ob['avg_composite_score'], oc['avg_composite_score'], od['avg_composite_score']),
    ("Reasoning Score", oa['avg_reasoning_score'], ob['avg_reasoning_score'], oc['avg_reasoning_score'], od['avg_reasoning_score']),
    ("Evidence Score", oa['avg_evidence_score'], ob['avg_evidence_score'], oc['avg_evidence_score'], od['avg_evidence_score']),
    ("Success%", oa['composite_success_rate']*100, ob['composite_success_rate']*100, oc['composite_success_rate']*100, od['composite_success_rate']*100),
    ("Success Rate", oa['success_rate']*100, ob['success_rate']*100, oc['success_rate']*100, od['success_rate']*100),
    ("Keyword Hit Rate", oa['avg_keyword_hit_rate']*100, ob['avg_keyword_hit_rate']*100, oc['avg_keyword_hit_rate']*100, od['avg_keyword_hit_rate']*100),
    ("Avg Time (s)", oa['avg_time_sec'], ob['avg_time_sec'], oc['avg_time_sec'], od['avg_time_sec']),
]

print(f"{'Metric':<25} {'A: RAG(C2)':>12} {'B: FT+prompt':>12} {'C: FT+RAG':>12} {'D: FT-bare':>12} {'B-A':>8} {'C-A':>8} {'D-A':>8} {'B-D':>8}")
print("-"*120)
for name, a, b, c, d in metrics:
    print(f"{name:<25} {a:>12.1f} {b:>12.1f} {c:>12.1f} {d:>12.1f} {b-a:>+8.1f} {c-a:>+8.1f} {d-a:>+8.1f} {b-d:>+8.1f}")

# C3目標達成判定
print(f"\n{'='*60}")
print(f"C3 Target Achievement (System C = FT+RAG)")
print(f"{'='*60}")

targets = [
    ("Composite Score", oc['avg_composite_score'], 75.0),
    ("Reasoning Score", oc['avg_reasoning_score'], 3.5),
    ("Evidence Score", oc['avg_evidence_score'], 4.0),
    ("Success%", oc['composite_success_rate']*100, 88.0),
]

all_met = True
for name, value, target in targets:
    met = value >= target
    if not met:
        all_met = False
    status = "PASS" if met else "MISS"
    print(f"  {name:<25} {value:>8.1f} / {target:.1f}  {status}")

print(f"\n  Overall: {'ALL TARGETS MET' if all_met else 'SOME TARGETS MISSED'}")

In [ ]:
# === レベル別比較 ===
print("="*120)
print("Level-wise Comparison")
print("="*120)

level_names = {1: "L1 Basic", 2: "L2 Spatial", 3: "L3 Constraint", 4: "L4 Decision", 5: "L5 Advanced"}

a_by_level = system_a_summary.get("by_level", {})
b_by_level = summary_b.get("by_level", {})
c_by_level = summary_c.get("by_level", {})
d_by_level = summary_d.get("by_level", {})

print(f"{'Level':<18} {'A: RAG(C2)':>12} {'B: FT+prompt':>12} {'C: FT+RAG':>12} {'D: FT-bare':>12} {'B-A':>8} {'C-A':>8} {'D-A':>8} {'Best':>10}")
print("-"*110)

for level in [1, 2, 3, 4, 5]:
    a_comp = a_by_level.get(level, a_by_level.get(str(level), {})).get('avg_composite_score', 0)
    b_comp = b_by_level.get(level, b_by_level.get(str(level), {})).get('avg_composite_score', 0)
    c_comp = c_by_level.get(level, c_by_level.get(str(level), {})).get('avg_composite_score', 0)
    d_comp = d_by_level.get(level, d_by_level.get(str(level), {})).get('avg_composite_score', 0)

    best = max([(a_comp, 'A'), (b_comp, 'B'), (c_comp, 'C'), (d_comp, 'D')], key=lambda x: x[0])
    print(f"{level_names[level]:<18} {a_comp:>12.1f} {b_comp:>12.1f} {c_comp:>12.1f} {d_comp:>12.1f} {b_comp-a_comp:>+8.1f} {c_comp-a_comp:>+8.1f} {d_comp-a_comp:>+8.1f} {best[1]:>10}")

In [ ]:
# === サブカテゴリ別比較 ===
print("="*120)
print("Subcategory-wise Comparison (Composite Score)")
print("="*120)

# System A のサブカテゴリ別スコアを結果から計算
from collections import defaultdict
a_by_subcat = defaultdict(list)
for r in system_a_raw:
    a_by_subcat[r['subcategory']].append(r['composite_score'])

b_by_subcat = defaultdict(list)
for r in results_b:
    b_by_subcat[r.subcategory].append(r.composite_score)

c_by_subcat = defaultdict(list)
for r in results_c:
    c_by_subcat[r.subcategory].append(r.composite_score)

d_by_subcat = defaultdict(list)
for r in results_d:
    d_by_subcat[r.subcategory].append(r.composite_score)

all_subcats = sorted(set(list(a_by_subcat.keys()) + list(b_by_subcat.keys()) + list(c_by_subcat.keys()) + list(d_by_subcat.keys())))

print(f"{'Subcategory':<25} {'N':>4} {'A: RAG':>10} {'B: FT+p':>10} {'C: FT+RAG':>10} {'D: bare':>10} {'C-A':>8} {'B-D':>8} {'Best':>8}")
print("-"*100)

for subcat in all_subcats:
    a_scores = a_by_subcat.get(subcat, [])
    b_scores = b_by_subcat.get(subcat, [])
    c_scores = c_by_subcat.get(subcat, [])
    d_scores = d_by_subcat.get(subcat, [])

    n = max(len(a_scores), len(b_scores), len(c_scores), len(d_scores))
    a_avg = np.mean(a_scores) if a_scores else 0
    b_avg = np.mean(b_scores) if b_scores else 0
    c_avg = np.mean(c_scores) if c_scores else 0
    d_avg = np.mean(d_scores) if d_scores else 0

    best = max([(a_avg, 'A'), (b_avg, 'B'), (c_avg, 'C'), (d_avg, 'D')], key=lambda x: x[0])
    print(f"{subcat:<25} {n:>4} {a_avg:>10.1f} {b_avg:>10.1f} {c_avg:>10.1f} {d_avg:>10.1f} {c_avg-a_avg:>+8.1f} {b_avg-d_avg:>+8.1f} {best[1]:>8}")

In [ ]:
# === 寄与分離分析 (4システム) ===
print("="*100)
print("Contribution Decomposition (4 Systems)")
print("="*100)

# C0ベースライン（非FT、非RAG）の想定値
C0_BASELINE = 52.2  # 既知のC0ベースラインスコア

decomp_metrics = [
    ("Composite", oa['avg_composite_score'], ob['avg_composite_score'], oc['avg_composite_score'], od['avg_composite_score']),
    ("Reasoning", oa['avg_reasoning_score'], ob['avg_reasoning_score'], oc['avg_reasoning_score'], od['avg_reasoning_score']),
    ("Evidence", oa['avg_evidence_score'], ob['avg_evidence_score'], oc['avg_evidence_score'], od['avg_evidence_score']),
    ("Success%", oa['composite_success_rate']*100, ob['composite_success_rate']*100, oc['composite_success_rate']*100, od['composite_success_rate']*100),
]

print(f"\n--- 1. 各要素の純粋効果 ---")
print(f"{'Metric':<15} {'FT純粋効果':>14} {'Prompt改善効果':>14} {'RAG効果(非FT)':>14} {'RAG効果(FT)':>14} {'FT+RAG相乗':>14}")
print(f"{'':15} {'(D-C0base)':>14} {'(B-D)':>14} {'(A-D)':>14} {'(C-B)':>14} {'(C-max)':>14}")
print("-"*90)

for name, a_val, b_val, c_val, d_val in decomp_metrics:
    if name == "Composite":
        ft_pure = d_val - C0_BASELINE       # FT単体の効果（C0プロンプト基準）
    else:
        ft_pure = float('nan')               # Composite以外はC0ベースラインなし
    prompt_effect = b_val - d_val            # プロンプト改善の効果
    rag_effect_non_ft = a_val - d_val        # RAG効果（非FTベース≒D基準）
    rag_effect_ft = c_val - b_val            # RAG効果（FTベース）
    synergy = c_val - max(a_val, b_val, d_val)  # 相乗効果

    if name == "Composite":
        print(f"{name:<15} {ft_pure:>+14.2f} {prompt_effect:>+14.2f} {rag_effect_non_ft:>+14.2f} {rag_effect_ft:>+14.2f} {synergy:>+14.2f}")
    else:
        print(f"{name:<15} {'N/A':>14} {prompt_effect:>+14.2f} {rag_effect_non_ft:>+14.2f} {rag_effect_ft:>+14.2f} {synergy:>+14.2f}")

print(f"\n--- 2. 寄与まとめ ---")
a_comp = oa['avg_composite_score']
b_comp = ob['avg_composite_score']
c_comp = oc['avg_composite_score']
d_comp = od['avg_composite_score']

print(f"  C0 ベースライン:           {C0_BASELINE:.1f}")
print(f"  D (FT-bare):               {d_comp:.1f}  (FT純粋効果: {d_comp - C0_BASELINE:+.1f})")
print(f"  B (FT+prompt):             {b_comp:.1f}  (プロンプト改善効果: {b_comp - d_comp:+.1f})")
print(f"  A (RAG, C2):               {a_comp:.1f}  (RAG+prompt効果: {a_comp - C0_BASELINE:+.1f})")
print(f"  C (FT+RAG):                {c_comp:.1f}  (全組合せ効果: {c_comp - C0_BASELINE:+.1f})")
print(f"  FT+RAG相乗効果:           {c_comp - max(a_comp, b_comp, d_comp):+.1f}  (C - max(A,B,D))")

print(f"\nInterpretation:")
print(f"  FT純粋効果 (D - C0baseline): FTモデルがC0プロンプトでどれだけ向上するか")
print(f"  プロンプト改善効果 (B - D): C1改善プロンプトの独自寄与")
print(f"  RAG効果・非FT (A - D): RAG+改善プロンプトの効果（FTなし）")
print(f"  RAG効果・FT (C - B): FTモデル上でのRAG追加効果")
print(f"  相乗効果 (C - max(A,B,D)): 全要素組合せによる追加効果")

In [ ]:
# === 過学習分析 ===
print("="*80)
print("Overfitting Analysis: FT-data vs Non-FT-data")
print("="*80)

for sys_label, results_list in [("B: FT+prompt", results_b), ("C: FT+RAG", results_c), ("D: FT-bare", results_d)]:
    ft_scores = [r.composite_score for r in results_list if r.test_id in ft_data_ids]
    non_ft_scores = [r.composite_score for r in results_list if r.test_id not in ft_data_ids]

    ft_avg = np.mean(ft_scores) if ft_scores else 0
    non_ft_avg = np.mean(non_ft_scores) if non_ft_scores else 0
    gap = ft_avg - non_ft_avg

    print(f"\n{sys_label}:")
    print(f"  FT data ({len(ft_scores)} cases):     avg composite = {ft_avg:.1f}")
    print(f"  Non-FT data ({len(non_ft_scores)} cases): avg composite = {non_ft_avg:.1f}")
    print(f"  Gap: {gap:+.1f}")

    if gap > 10:
        print(f"  \u26a0\ufe0f SIGNIFICANT overfitting detected (gap > 10pt)")
    elif gap > 5:
        print(f"  \u26a0\ufe0f Moderate overfitting (gap 5-10pt)")
    else:
        print(f"  \u2705 No significant overfitting (gap <= 5pt)")

# System Aも比較用に表示
a_ft_scores = [r['composite_score'] for r in system_a_raw if r['test_id'] in ft_data_ids]
a_non_ft_scores = [r['composite_score'] for r in system_a_raw if r['test_id'] not in ft_data_ids]
print(f"\nA: RAG(C2) (reference):")
print(f"  FT data ({len(a_ft_scores)} cases):     avg composite = {np.mean(a_ft_scores):.1f}")
print(f"  Non-FT data ({len(a_non_ft_scores)} cases): avg composite = {np.mean(a_non_ft_scores):.1f}")
print(f"  Gap: {np.mean(a_ft_scores) - np.mean(a_non_ft_scores):+.1f}")

In [ ]:
# === C2弱点改善分析 ===
print("="*80)
print("C2 Weakness Improvement: Cases with C2 composite 50-70")
print("="*80)

# C2で50-70点だったケース
weak_cases = {r['test_id']: r['composite_score'] for r in system_a_raw
              if 50 <= r['composite_score'] < 70}

print(f"Weak cases (C2 composite 50-70): {len(weak_cases)}")

b_weak = {r.test_id: r.composite_score for r in results_b if r.test_id in weak_cases}
c_weak = {r.test_id: r.composite_score for r in results_c if r.test_id in weak_cases}
d_weak = {r.test_id: r.composite_score for r in results_d if r.test_id in weak_cases}

improvements_b = sum(1 for tid in weak_cases if b_weak.get(tid, 0) > weak_cases[tid])
improvements_c = sum(1 for tid in weak_cases if c_weak.get(tid, 0) > weak_cases[tid])
improvements_d = sum(1 for tid in weak_cases if d_weak.get(tid, 0) > weak_cases[tid])

b_deltas = [b_weak.get(tid, 0) - weak_cases[tid] for tid in weak_cases]
c_deltas = [c_weak.get(tid, 0) - weak_cases[tid] for tid in weak_cases]
d_deltas = [d_weak.get(tid, 0) - weak_cases[tid] for tid in weak_cases]

print(f"\nSystem B (FT+prompt):")
print(f"  Improved: {improvements_b}/{len(weak_cases)} ({improvements_b/len(weak_cases)*100:.1f}%)")
print(f"  Avg delta: {np.mean(b_deltas):+.1f}")

print(f"\nSystem C (FT+RAG):")
print(f"  Improved: {improvements_c}/{len(weak_cases)} ({improvements_c/len(weak_cases)*100:.1f}%)")
print(f"  Avg delta: {np.mean(c_deltas):+.1f}")

print(f"\nSystem D (FT-bare):")
print(f"  Improved: {improvements_d}/{len(weak_cases)} ({improvements_d/len(weak_cases)*100:.1f}%)")
print(f"  Avg delta: {np.mean(d_deltas):+.1f}")

## 8. 可視化

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

# --- Fig 1: 4 System Overall Comparison ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

systems = ['A: RAG\n(C2)', 'B: FT+prompt', 'C: FT+RAG', 'D: FT-bare']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

# Composite Score
comp_scores = [oa['avg_composite_score'], ob['avg_composite_score'], oc['avg_composite_score'], od['avg_composite_score']]
bars = axes[0].bar(systems, comp_scores, color=colors, alpha=0.8)
axes[0].set_ylabel('Composite Score')
axes[0].set_title('Composite Score Comparison')
axes[0].set_ylim([0, 100])
axes[0].axhline(y=75, color='red', linestyle='--', alpha=0.5, label='C3 Target (75)')
axes[0].legend()
for bar, val in zip(bars, comp_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}', ha='center', fontweight='bold')

# Multi-dimensional (normalized to 0-100)
metric_names = ['Reasoning\n(x20)', 'Evidence\n(x20)', 'Success%']
a_vals = [oa['avg_reasoning_score']*20, oa['avg_evidence_score']*20, oa['composite_success_rate']*100]
b_vals = [ob['avg_reasoning_score']*20, ob['avg_evidence_score']*20, ob['composite_success_rate']*100]
c_vals = [oc['avg_reasoning_score']*20, oc['avg_evidence_score']*20, oc['composite_success_rate']*100]
d_vals = [od['avg_reasoning_score']*20, od['avg_evidence_score']*20, od['composite_success_rate']*100]

x = np.arange(len(metric_names))
width = 0.20

axes[1].bar(x - 1.5*width, a_vals, width, label='A: RAG(C2)', color=colors[0], alpha=0.8)
axes[1].bar(x - 0.5*width, b_vals, width, label='B: FT+prompt', color=colors[1], alpha=0.8)
axes[1].bar(x + 0.5*width, c_vals, width, label='C: FT+RAG', color=colors[2], alpha=0.8)
axes[1].bar(x + 1.5*width, d_vals, width, label='D: FT-bare', color=colors[3], alpha=0.8)

axes[1].set_ylabel('Score (0-100 scale)')
axes[1].set_title('Multi-dimensional Score Comparison')
axes[1].set_xticks(x)
axes[1].set_xticklabels(metric_names)
axes[1].legend()
axes[1].set_ylim([0, 100])

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/phase9c_step3_overall_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- Fig 2: Level-wise Comparison ---
fig, ax = plt.subplots(figsize=(16, 6))

levels = [1, 2, 3, 4, 5]
level_labels = ['L1\nBasic', 'L2\nSpatial', 'L3\nConstraint', 'L4\nDecision', 'L5\nAdvanced']

a_scores = [a_by_level.get(l, a_by_level.get(str(l), {})).get('avg_composite_score', 0) for l in levels]
b_scores_l = [b_by_level.get(l, b_by_level.get(str(l), {})).get('avg_composite_score', 0) for l in levels]
c_scores_l = [c_by_level.get(l, c_by_level.get(str(l), {})).get('avg_composite_score', 0) for l in levels]
d_scores_l = [d_by_level.get(l, d_by_level.get(str(l), {})).get('avg_composite_score', 0) for l in levels]

x = np.arange(len(levels))
width = 0.20

ax.bar(x - 1.5*width, a_scores, width, label='A: RAG(C2)', color='#3498db', alpha=0.8)
ax.bar(x - 0.5*width, b_scores_l, width, label='B: FT+prompt', color='#e74c3c', alpha=0.8)
ax.bar(x + 0.5*width, c_scores_l, width, label='C: FT+RAG', color='#2ecc71', alpha=0.8)
ax.bar(x + 1.5*width, d_scores_l, width, label='D: FT-bare', color='#f39c12', alpha=0.8)

ax.set_xlabel('Level')
ax.set_ylabel('Composite Score')
ax.set_title('4 System Comparison by Difficulty Level')
ax.set_xticks(x)
ax.set_xticklabels(level_labels)
ax.legend()
ax.set_ylim([0, 100])
ax.axhline(y=75, color='red', linestyle='--', alpha=0.3, label='C3 Target')

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/phase9c_step3_level_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- Fig 3: Overfitting Analysis ---
fig, ax = plt.subplots(figsize=(12, 6))

systems_label = ['A: RAG(C2)', 'B: FT+prompt', 'C: FT+RAG', 'D: FT-bare']

# Compute FT-data vs non-FT-data scores
ft_avgs = []
non_ft_avgs = []

for results_list, is_raw in [(system_a_raw, True), (results_b, False), (results_c, False), (results_d, False)]:
    if is_raw:
        ft = [r['composite_score'] for r in results_list if r['test_id'] in ft_data_ids]
        non_ft = [r['composite_score'] for r in results_list if r['test_id'] not in ft_data_ids]
    else:
        ft = [r.composite_score for r in results_list if r.test_id in ft_data_ids]
        non_ft = [r.composite_score for r in results_list if r.test_id not in ft_data_ids]
    ft_avgs.append(np.mean(ft) if ft else 0)
    non_ft_avgs.append(np.mean(non_ft) if non_ft else 0)

x = np.arange(len(systems_label))
width = 0.35

bars1 = ax.bar(x - width/2, ft_avgs, width, label=f'FT data ({len(ft_data_ids)} cases)', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, non_ft_avgs, width, label=f'Non-FT data ({130-len(ft_data_ids)} cases)', color='#3498db', alpha=0.8)

ax.set_ylabel('Avg Composite Score')
ax.set_title('Overfitting Analysis: FT-data vs Non-FT-data')
ax.set_xticks(x)
ax.set_xticklabels(systems_label)
ax.legend()
ax.set_ylim([0, 100])

# Gap labels
for i in range(len(systems_label)):
    gap = ft_avgs[i] - non_ft_avgs[i]
    max_val = max(ft_avgs[i], non_ft_avgs[i])
    ax.text(i, max_val + 2, f'gap={gap:+.1f}', ha='center', fontweight='bold',
            color='red' if abs(gap) > 5 else 'black')

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/phase9c_step3_overfitting.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. 結果保存

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

def convert_to_serializable(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(i) for i in obj]
    return obj

# 結果データ構築
results_data = {
    "experiment": "phase9c_step3",
    "description": "QLoRA FineTuning evaluation (4 system comparison: RAG vs FT+prompt vs FT+RAG vs FT-bare)",
    "timestamp": timestamp,
    "test_count": 130,
    "model": "Qwen/Qwen3-32B",
    "adapter": "qwen3-32b-poi-qlora",
    "training_data_count": len(ft_data_ids),
    "systems": {
        "A_rag_c2": {
            "description": "RAG (C2): Qwen3-32B + RAG (existing results)",
            "summary": convert_to_serializable(system_a_summary),
            "source": "results/phase9c_step2_20260227_071313.json",
        },
        "B_ft_only": {
            "description": "FT+prompt: QLoRA Qwen3-32B + C1 improved system prompt (no RAG)",
            "summary": convert_to_serializable(summary_b),
            "results": [convert_to_serializable(r.to_dict()) for r in results_b],
        },
        "C_ft_rag": {
            "description": "FT+RAG: QLoRA Qwen3-32B + RAG",
            "summary": convert_to_serializable(summary_c),
            "results": [convert_to_serializable(r.to_dict()) for r in results_c],
        },
        "D_ft_bare": {
            "description": "FT-bare: QLoRA Qwen3-32B + C0 original prompt (no RAG)",
            "summary": convert_to_serializable(summary_d),
            "results": [convert_to_serializable(r.to_dict()) for r in results_d],
        },
    },
    "overfitting_analysis": {
        "ft_data_ids_count": len(ft_data_ids),
        "non_ft_data_count": 130 - len(ft_data_ids),
    },
}

output_file = f"{PROJECT_PATH}/results/phase9c_step3_{timestamp}.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results_data, f, ensure_ascii=False, indent=2)

print(f"Results saved to: {output_file}")
print(f"File size: {os.path.getsize(output_file) / 1024:.1f} KB")

## 10. 結論

### 4システム比較結果

| System | Composite | Reasoning | Evidence | Success% |
|--------|-----------|-----------|----------|----------|
| A: RAG (C2) | 70.4 | 3.07 | 3.85 | 83.1% |
| B: FT+prompt | **TBD** | **TBD** | **TBD** | **TBD** |
| C: FT+RAG | **TBD** | **TBD** | **TBD** | **TBD** |
| D: FT-bare | **TBD** | **TBD** | **TBD** | **TBD** |

### 寄与分離

| 要素 | 効果 | 測定方法 |
|------|------|----------|
| FT純粋効果 | **TBD** | D - C0baseline (52.2) |
| プロンプト改善効果 | **TBD** | B - D |
| RAG効果 | **TBD** | C - B |
| FT+RAG相乗効果 | **TBD** | C - max(A,B,D) |

### C3目標達成

| 指標 | C3目標 | System C | 判定 |
|------|--------|----------|------|
| composite_score | 75+ | **TBD** | **TBD** |
| reasoning_score | 3.5+ | **TBD** | **TBD** |
| evidence_score | 4.0+ | **TBD** | **TBD** |
| composite_success_rate | 88%+ | **TBD** | **TBD** |

## 11. 結果をGitにコミット & プッシュ

In [ ]:
# 結果をプッシュ
!git -C {REPO_PATH} add notebooks/phase9c_step3_evaluation.ipynb
!git -C {REPO_PATH} add results/phase9c_step3_*
!git -C {REPO_PATH} add results/checkpoint_c3_*

!git -C {REPO_PATH} diff --cached --stat

!git -C {REPO_PATH} commit -m "results: Phase 9-C Step 3 QLoRA 4システム比較評価結果"
!git -C {REPO_PATH} push origin {BRANCH}

print("\n\u2705 評価結果をリモートブランチにプッシュしました")